In [1]:
import ctadata
from astropy.io import fits
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord

In [2]:
!cta-data start-agent

/layers/paketo-buildpacks_conda-env-update/conda-env/bin/oidc-agent
/layers/paketo-buildpacks_conda-env-update/conda-env/bin/davix-ls
/layers/paketo-buildpacks_conda-env-update/conda-env/bin/davix-get
/layers/paketo-buildpacks_conda-env-update/conda-env/bin/davix-put
success


In [26]:
head_data_path = "../cta_dc_data/"
irfs_data_path = head_data_path + "irfs/"
mrk501_data_path = head_data_path + "mrk_501/"
es0229_data_path = head_data_path + "es0229/"

In [23]:
!mkdir -p {irfs_data_path}
!mkdir -p {mrk501_data_path}
!mkdir -p {es0229_data_path}

In [5]:
ctadata.fetch_and_save_file(f'pnfs/cta.cscs.ch/cta/sdc-internal-data/obs-index.fits.gz')
!mv obs-index.fits.gz {head_data_path}

In [6]:
obs_list = fits.open(head_data_path + "obs-index.fits.gz")[1]
obs_ra = np.array(obs_list.data["RA_PNT"])
obs_dec = np.array(obs_list.data["DEC_PNT"])
obs_coord = SkyCoord(ra=obs_ra*u.deg, dec=obs_dec*u.deg, frame='icrs')

mrk501_coord = SkyCoord.from_name("Mrk 501")
es0229_coord = SkyCoord.from_name("1ES 0229+200")

In [7]:
max_separation = 2
mrk501_obs_mask = obs_coord.separation(mrk501_coord).degree < max_separation
es0229_obs_mask = obs_coord.separation(es0229_coord).degree < max_separation

In [36]:
irf_files = set()

for file in obs_list.data["IRF_FILENAME"][mrk501_obs_mask]:
    irf_files.add(file)
for file in obs_list.data["IRF_FILENAME"][es0229_obs_mask]:
    irf_files.add(file)
for file in irf_files:
    print(str(file))

Prod5-North-40deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz
Prod5-North-20deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz


In [40]:
# If the link has not changed, this should download directly the IRF compressed folder from zenodo,
# and decompress it to retrieve the two files needed.
!mkdir -p tmp && cd tmp && \
wget https://zenodo.org/records/5499840/files/cta-prod5-zenodo-fitsonly-v0.1.zip?download=1 -O cta-prod5-zenodo-fitsonly-v0.1.zip && \
unzip cta-prod5-zenodo-fitsonly-v0.1.zip && \
tar -xzf fits/CTA-Performance-prod5-v0.1-North-20deg.FITS.tar.gz && \
tar -xzf fits/CTA-Performance-prod5-v0.1-North-40deg.FITS.tar.gz

for irf_file in irf_files:
    !mv tmp/{irf_file} {irfs_data_path}

!rm -r tmp

--2026-02-26 14:18:55--  https://zenodo.org/records/5499840/files/cta-prod5-zenodo-fitsonly-v0.1.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.184.103.118, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44065624 (42M) [application/octet-stream]
Saving to: ‘cta-prod5-zenodo-fitsonly-v0.1.zip’

cta-prod5-zenodo-fi 100%[===================>]  42.02M  3.25MB/s    in 18s     

2026-02-26 14:19:13 (2.35 MB/s) - ‘cta-prod5-zenodo-fitsonly-v0.1.zip’ saved [44065624/44065624]

Archive:  cta-prod5-zenodo-fitsonly-v0.1.zip
  inflating: LICENSE                 
  inflating: README.md               
  inflating: Website.md              
  inflating: figures/CTA-Performance-prod5-v0.1-North-Alpha-EffectiveArea-DirectionCuts.png  
  inflating: figures/CTA-Performance-prod5-v0.1-North-Alpha-EffectiveArea-NoDirectionCuts.png  
  inflating: figures/CTA-Performance-prod5-v0.1-No

In [9]:
for obs_id in obs_list.data["OBS_ID"][mrk501_obs_mask]:
    ctadata.fetch_and_save_file(f'pnfs/cta.cscs.ch/cta/sdc-internal-data/products_SDC/Events/North/events_{obs_id}.fits')
    !mv events_{obs_id}.fits mrk501_data_path

for obs_id in obs_list.data["OBS_ID"][es0229_obs_mask]:
    ctadata.fetch_and_save_file(f'pnfs/cta.cscs.ch/cta/sdc-internal-data/products_SDC/Events/North/events_{obs_id}.fits')
    !mv events_{obs_id}.fits es0229_data_path